<a href="https://colab.research.google.com/github/ParkHangah/AIFFEL_quest_eng/blob/master/LLM_Aplication/LLM03/HuggingFaceProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 한국어 버전 KLUE benchmark 을 이용한 커스텀 프로젝트 직접 만들기


## 환경구성

In [ ]:
# 라이브러리 버전 확인
import tensorflow
import numpy
import transformers
import datasets

print(tensorflow.__version__)
print(numpy.__version__)
print(transformers.__version__)
print(datasets.__version__)

2.19.0
2.0.2
5.0.0
4.0.0


In [ ]:
# Huggingface transformers 설치 및 환경 구성
! pip install transformers
! pip install accelerate

## STEP 1. NSMC 데이터 분석 및 Huggingface dataset 구성

#### 1)  Google 드라이브 연결

##### (1) 경로 설정

In [ ]:
# 드라이브 경로 설정  by 박항아
drive_path = '#Study/Aiffel/Work' # 평상시 작업하는 드라이브 폴더 경로를 입력해 주세요.
project_name = 'Huggingface'       # 이번 프로젝트세 사용하는 폴더명을 입력해주세요.

##### (2) 드라이브 연결

In [ ]:
from google.colab import drive
from IPython.display import clear_output
import os

# 1. 구글 드라이브 마운트
print("Connecting...")
drive.mount('/content/gdrive')

# 2. 경로 설정 및 폴더 생성
base_path = os.path.join('/content/gdrive/MyDrive',drive_path)
project_path = os.path.join(base_path, project_name)

# Create the project directory if it doesn't exist
os.makedirs(project_path, exist_ok=True)


# 2. 출력 전체 삭제 후 완료 메시지 출력
clear_output()
print("✅ 구글 드라이브 연결이 성공적으로 완료되었습니다.")
print(f"Selected Google Drive root path: {base_path}")

✅ 구글 드라이브 연결이 성공적으로 완료되었습니다.
Selected Google Drive root path: /content/gdrive/MyDrive/#Study/Aiffel/Work


#### 2) 커스템 데이터셋 만들기

In [ ]:
# 1. 필요한 라이브러리 임포트
import pandas as pd
import os  # os.path 사용을 위해 추가가 필요합니다.
from datasets import Dataset
# 2. 데이터셋을 파싱할 함수를 선언
def parse_nsmc_file(file_path):
    """
    NSMC 원본 텍스트 파일은 탭(\t)으로 구분된 TSV 형식입니다.
    이 함수는 각 줄을 읽어 데이터 구조에 맞게 안전하게 파싱합니다.
    """
    # 2-1. 데이터를 담을 빈 딕셔너리 구조를 미리 정의
    data = {
        'id': [],      # 문장의 고유 ID
        'document': [],  # 문장 텍스트
        'label': [],    # 감정분석 (1: 긍정, 0: 부정)
        }

    # 2-2. 파일을 UTF-8 인코딩으로 읽기 모드로 엽니다.
    with open(file_path, 'r', encoding='utf-8') as f:
        # 첫 번째 줄(헤더: Quality, #1 ID 등 컬럼명)은 데이터가 아니므로 스킵합니다.
        next(f)

        # 파일을 한 줄씩 읽으며 처리
        # enumerate를 써서 오류 발생 시 줄 번호를 확인합니다.
        for line_num, line in enumerate(f, 1):
            try:
                # 탭(\t)을 기준으로 한 문장을 나누기.
                parts = line.strip().split('\t')
                # 정상적인 데이터라면 최소 3개 이상의 컬럼이 있어야 함
                if len(parts) >= 3:
                    # 3개 컬럼으로 분할 (
                    # 각 인덱스에 맞는 값을 추출하여 형 변환을 수행.
                    id = int(parts[0])
                    document = '\t'.join(parts[1:-1])
                    label = int(parts[-1])

                    # 파싱된 데이터를 딕셔너리에 추가
                    data['id'].append(id)
                    data['document'].append(document)
                    data['label'].append(label)
                else:
                    # 컬럼 개수가 부족한 잘못된 형식의 줄은 건너뜁니다.
                    print(f"Line {line_num}: Invalid format, skipping")
            except Exception as e:
                # 예외 발생 시(예: 숫자로 변환 불가능한 값 등) 로그를 남기고 넘어가기.
                print(f"Line {line_num}: Error {e}, skipping")

    # 2-3. 수집된 딕셔너리를 최종적으로 Pandas DataFrame으로 변환하여 반합니다.
    return pd.DataFrame(data)
# 3. 로컬 파일에서 NSMC 데이터 읽기
train_df = parse_nsmc_file(os.path.join(project_path, 'data/nsmc_ratings_train.txt'))
test_df = parse_nsmc_file(os.path.join(project_path, 'data/nsmc_ratings_test.txt'))

In [ ]:
# 데이터 구조 확인
print("Train dataset columns:", train_df.columns.tolist())
print("Train dataset shape:", train_df.shape)
print("\nFirst 5 examples:")

for i in range(5):
    row = train_df.iloc[i]
    for col in train_df.columns:
        print(f"{col}: {row[col]}")
    print('\n')

Train dataset columns: ['id', 'document', 'label']
Train dataset shape: (150000, 3)

First 5 examples:
id: 9976970
document: 아 더빙.. 진짜 짜증나네요 목소리
label: 0


id: 3819312
document: 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
label: 1


id: 10265843
document: 너무재밓었다그래서보는것을추천한다
label: 0


id: 9045019
document: 교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
label: 0


id: 6483659
document: 사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다
label: 1




#### 3) custom dataset을 재구성

In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
# 1. Pandas DataFrame을 파이썬 리스트를 담은 딕셔너리 형식으로 변환
# 허깅페이스의 Dataset.from_dict() 함수에 넣기 위한 준비 단계
train_dataset = train_df.to_dict('list')
test_dataset = test_df.to_dict('list')
# 2. 데이터를 학습(Train)과 검증(Validation) 세트로 분할하기 위해 인덱스를 생성
train_indices = list(range(len(train_dataset['label'])))
# 3. 사이킷런의 train_test_split을 사용하여 데이터를 8:2 비율로 분할
# [중요] stratify=train_dataset['label']: 정답(라벨)의 비율이
# 학습셋과 검증셋에 동일하게 유지되도록 하여 데이터 불균형 문제를 방지
train_idx, val_idx = train_test_split(
    train_indices,
    test_size=0.2,
    random_state=42,
    stratify=train_dataset['label']
)
# 4. 검증(Validation) 데이터셋 구축
# 분리된 인덱스(val_idx)에 해당하는 데이터만 추출하여 새로운 딕셔너리 생성
validation_dataset = {}
for key in train_dataset.keys():
    validation_dataset[key] = [train_dataset[key][i] for i in val_idx]
# 5. 최종 학습(Train) 데이터셋 업데이트
# 검증셋으로 빠진 데이터를 제외하고 학습에만 사용할 데이터들로 구성
train_dataset_final = {}
for key in train_dataset.keys():
    train_dataset_final[key] = [train_dataset[key][i] for i in train_idx]
# 6. 파이썬 딕셔너리 데이터를 허깅페이스 전용 'Dataset' 객체로 변환
# 허깅페이스의 다양한 도구(Trainer, Tokenizer 등)와 호환성 확보
train_hf_dataset = Dataset.from_dict(train_dataset_final)
validation_hf_dataset = Dataset.from_dict(validation_dataset)
test_hf_dataset = Dataset.from_dict(test_dataset)
# 7. Dataset 객체를 하나로 묶어 'DatasetDict'를 생성 (허깅페이스 표준)
customized_mrpc_dataset = DatasetDict({
    'train': train_hf_dataset,
    'validation': validation_hf_dataset,
    'test': test_hf_dataset
})
#  8. 결과 출력
print("DatasetDict({")
for split_name, split_data in customized_mrpc_dataset.items():
    # 각 분할(train/val/test)의 컬럼명과 행 수를 출력하여 구조를 확인.
    print(f"    {split_name}: Dataset({{")
    print(f"        features: {list(split_data.features.keys())},")
    print(f"        num_rows: {split_data.num_rows}")
    print("    })")
print("})")

print("\n데이터셋 정보 확인!")
# num_rows와 features 속성을 통해 데이터셋의 형태(Shape)를 파악.
print(f"Train dataset shape: ({customized_mrpc_dataset['train'].num_rows}, {len(customized_mrpc_dataset['train'].features)})")
print(f"Validation dataset shape: ({customized_mrpc_dataset['validation'].num_rows}, {len(customized_mrpc_dataset['validation'].features)})")
print(f"Test dataset shape: ({customized_mrpc_dataset['test'].num_rows}, {len(customized_mrpc_dataset['test'].features)})")

print("Dataset 생성 완료!")

# 9. 최종적으로 첫 번째 학습 샘플을 출력하여 데이터가 의도대로 들어갔는지 검토
print(f"\n첫 번째 train 샘플:")
print(customized_mrpc_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 120000
    })
    validation: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})

데이터셋 정보 확인!
Train dataset shape: (120000, 3)
Validation dataset shape: (30000, 3)
Test dataset shape: (50000, 3)
Dataset 생성 완료!

첫 번째 train 샘플:
{'id': 10238686, 'document': '일본이 투자한듯~~~', 'label': 0}


## STEP 2. klue/bert-base model 및 tokenizer 불러오기

#### 1) SequenceClassification 모델에 맞는 토크나이저와 모델을 로드

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 모델 경로(Hugging Face 허브 ID)를 지정합니다.
model_name = "monologg/koelectra-base-v3-discriminator"

# 2. 토크나이저와 모델 로드하기
huggingface_tokenizer = AutoTokenizer.from_pretrained(model_name)
huggingface_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 2)

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missin

#### 2) 토크나이징을 위한 transform 함수 선언


In [ ]:
def transform_nsmc(batch):
    # NSMC 데이터셋의 문장 컬럼인 'document' 하나만 전달합니다.
    return huggingface_tokenizer(
        batch['document'],
        truncation=True,
        padding='max_length',
        max_length=64, # NSMC 리뷰는 길지 않으므로 64~128 정도면 충분합니다.
        return_token_type_ids=False, # ELECTRA도 DistilBERT처럼 필수 사항은 아닙니다.
    )

#### 3) map을 이용해 데이터셋을 한번에 토크나이징 (batch를 적용 필수)

In [ ]:
hf_dataset = customized_mrpc_dataset.map(transform_nsmc, batched=True)
# train & validation & test split
hf_train_dataset = hf_dataset['train']
hf_val_dataset = hf_dataset['validation']
hf_test_dataset = hf_dataset['test']

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

## 3.Train/Evaluation과 Test

In [ ]:
import os
import numpy as np
from transformers import Trainer, TrainingArguments
# 1. 훈련 결과물(체크포인트, 모델 설정 등)이 저장될 폴더 이름을 지정
output_dir = os.path.join(project_path, 'nsmc_transformers')
# 2. 모델 학습을 위한 상세 설정(Hyperparameters)을 정의
training_arguments = TrainingArguments(
    output_dir,					# output이 저장될 경로
    eval_strategy="epoch",           		# evaluation하는 빈도
    learning_rate = 2e-5,			# learning_rate
    per_device_train_batch_size = 8, 	# 각 device 당 batch size
    per_device_eval_batch_size = 8,	# evaluation 시에 batch size
    num_train_epochs = 3,			# train 시킬 총 epochs
    weight_decay = 0.01, 			# weight decay
)

In [ ]:
# 3. 모델의 성능을 평가할 지표를 계산하기 위한 라이브러리를 설치하고 로드
# Trainer 시 인자로 넘겨주어야 할 것 중에 compute_metrics 메소드가 있음
# task가 classification인지 regression인지에 따라 모델의 출력 형태가 달라지므로 task별로 적합한 출력 형식을 고려해 모델의 성능을 계산 하는 방법을 미리 지정
!pip install evaluate
from evaluate import load
# 2. 개별 평가지표 로드 (정확도와 F1 스코어) # NSMC는 데이터가 균형 잡혀 있어 Accuracy가 주 지표이지만, # 모델의 정밀함을 확인하기 위해 F1도 함께 보는 것이 정석입니다.
accuracy_metric = load("accuracy")
f1_metric = load("f1")

In [ ]:
# 4. 모델의 예측값을 받아서 실제 점수로 변환해주는 함수를 정의
def compute_metrics(eval_pred):
    """
    모델의 출력(Logits)을 정답 라벨과 비교하여 정확도와 F1을 계산합니다.
    """
    logits, labels = eval_pred
    # 80% 확률로 긍정(1), 20%로 부정(0)이라면 -> 더 큰 값인 '1'을 예측값으로 선택
    predictions = np.argmax(logits, axis=1)
    # 정확도 계산
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    # F1 스코어 계산 (이진 분류이므로 binary 설정)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    # 두 지표를 합쳐서 반환 (Trainer가 이걸 보고 매 에폭 결과를 출력함)
    return { "accuracy": acc["accuracy"], "f1": f1["f1"] }


In [ ]:
# 5. Trainer 인스턴스 생성 후  train을 진행
trainer = Trainer(
    model=huggingface_model, 	    # 학습시킬 model
    args=training_arguments,	    # TrainingArguments을 통해 설정한 arguments
    train_dataset=hf_train_dataset,	    # trainiTrainer 인스턴스 호출ng dataset
    eval_dataset=hf_val_dataset,	    # evaluation dataset
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.299062,0.271511,0.899933,0.899099
2,0.225958,0.342808,0.904867,0.906414
3,0.168262,0.428027,0.906400,0.906915


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=45000, training_loss=0.2404640875922309, metrics={'train_runtime': 3382.8934, 'train_samples_per_second': 106.418, 'train_steps_per_second': 13.302, 'total_flos': 1.18399974912e+16, 'train_loss': 0.2404640875922309, 'epoch': 3.0})

##### 3) test 데이터셋으로 평가


In [ ]:
trainer.evaluate(hf_test_dataset)

{'eval_loss': 0.42433011531829834,
 'eval_accuracy': 0.90652,
 'eval_f1': 0.9077196446199408,
 'eval_runtime': 85.1213,
 'eval_samples_per_second': 587.397,
 'eval_steps_per_second': 73.425,
 'epoch': 3.0}

## STEP 4. Fine-tuning을 통하여 모델 성능(accuarcy) 향상시키기

### [방법 1]  하이퍼파라미터 최적화

- Batch Size 확대: 이전 코드에서 8을 쓰셨다면, 이를 **32 또는 64**로 늘려보세요. 배치가 클수록 그래디언트가 안정되어 학습이 더 잘 됩니다. (메모리 부족 시 gradient_accumulation_steps 활용)

- Learning Rate 조정: 2e-5에서 시작하되, 성능이 정체되면 3e-5나 5e-5로 살짝 높여보세요.

- Epochs: NSMC는 데이터가 많아 3~5 Epoch 정도면 충분합니다. 너무 많으면 과적합(Overfitting)되니 주의하세요.

### [방법 2] 데이터 정제 (Data Cleaning)*이탤릭체 텍스트*

- 중복 제거: train_df.drop_duplicates(subset=['document'])는 필수입니다.

- 너무 짧은 리뷰 제거: "아", "음" 같은 1~2글자 리뷰는 노이즈가 될 수 있습니다. 5글자 미만은 과감히 쳐내는 것도 방법입니다.

- Max Length 최적화: max_length=64 또는 128이 적당합니다. 너무 길면 불필요한 패딩(Padding) 때문에 학습이 느려지고 노이즈가 낍니다.

## STEP 5. Bucketing을 적용하여 학습시키고, STEP 4의 결과와의 비교

- Dynamic Padding: 배치를 만들 때, 해당 배치 안에서 가장 긴 문장에 맞춰 패딩을 넣는 방식입니다.

- Bucketing (group_by_length): 비슷한 길이의 문장끼리 묶어서 배치를 만듭니다. 이렇게 하면 짧은 문장끼리 모인 배치는 패딩이 거의 없어 학습 속도가 비약적으로 빨라짐

## 시간관계상 STEP 4와 5를 같이 적용

In [ ]:
import torch
import gc

# 1. 기존 모델 변수가 있다면 제거
if 'huggingface_model' in globals():
    del huggingface_model

# 2. 가비지 컬렉션을 강제로 실행하여 메모리 파편을 정리
gc.collect()

# 3. GPU 메모리(VRAM) 내의 캐시를 비워줍니다.
torch.cuda.empty_cache()

print("GPU 메모리 정리가 완료되었습니다!")

GPU 메모리 정리가 완료되었습니다!


In [ ]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

# 1. Dynamic Padding을 위한 Data Collator 선언
# 이 객체는 각 배치 내에서 가장 긴 문장의 길이에 맞춰 실시간으로 패딩을 채워줍니다.
data_collator = DataCollatorWithPadding(tokenizer=huggingface_tokenizer)

In [ ]:
# 2. 모델 저장경로 및 하이퍼파라미터 재설정 (배치사이즈를 8에서 32로 증대)
output_dir_batch32 = os.path.join(project_path, 'nsmc_transformers_batch32')
training_arguments = TrainingArguments(
    output_dir,					# output이 저장될 경로
    eval_strategy="epoch",           		# evaluation하는 빈도
    learning_rate = 2e-5,			# learning_rate
    per_device_train_batch_size = 32, 	# 각 device 당 batch size
    per_device_eval_batch_size = 32,	# evaluation 시에 batch size
    num_train_epochs = 3,			# train 시킬 총 epochs
    weight_decay = 0.01, 			# weight decay
    group_by_length=True,
    logging_steps=100,
)

In [ ]:
# 3. 모델 초기화
from transformers import AutoModelForSequenceClassification
huggingface_model_step5 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missin

In [ ]:
# 4. Trainer 정의
trainer_step5 = Trainer(
    model=huggingface_model_step5,
    args=training_arguments,
    train_dataset=hf_train_dataset,	  # trainiTrainer 인스턴스 호출ng dataset
    eval_dataset=hf_val_dataset,	    # evaluation dataset
    data_collator=data_collator,      # [중요] Dynamic Padding 적용
    compute_metrics=compute_metrics,
)

# 5. 학습 시작 및 시간 측정
print("Step 5 Bucketing 학습 시작...")
trainer_step5.train()

Step 5 Bucketing 학습 시작...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.235924,0.248634,0.904667,0.904901
2,0.201652,0.236843,0.908167,0.908053
3,0.130800,0.285101,0.908967,0.910079


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11250, training_loss=0.20388064024183486, metrics={'train_runtime': 2527.3024, 'train_samples_per_second': 142.444, 'train_steps_per_second': 4.451, 'total_flos': 1.18399974912e+16, 'train_loss': 0.20388064024183486, 'epoch': 3.0})

In [ ]:
# test 데이터셋으로 평가
trainer.evaluate(hf_test_dataset)

{'eval_loss': 0.42433011531829834,
 'eval_accuracy': 0.90652,
 'eval_f1': 0.9077196446199408,
 'eval_runtime': 84.469,
 'eval_samples_per_second': 591.933,
 'eval_steps_per_second': 73.992,
 'epoch': 3.0}

## [보고서] NSMC 감성 분석 모델 Fine-tuning 결과

#### 1. 실험 개요
- 사용 모델: monologg/koelectra-base-v3-discriminator
- 데이터셋: NSMC (Naver Sentiment Movie Corpus)
- 적용 기술: Batch Size 상향(32), Bucketing(group_by_length), Dynamic Padding


#### 2. 학습결과

##### 2-1. 최초학습 : Epoch 3 / Batch Size: 8

| Epoch | Training Loss | Validation Loss | Accuracy | F1 Score |
| :---: | :---: | :---: | :---: | :---: |
| **1** | 0.2991 | 0.2715 | 0.8999 | 0.8991 |
| **2** | 0.2260 | 0.3428 | 0.9049 | 0.9064 |
| **3** | 0.1683 | 0.4280 | 0.9064 | 0.9069 |

##### 2-2. step4,5적용 (1) : Epoch 3 / Batch Size: 32, Dynamic Padding 적용

| Epoch | Training Loss | Validation Loss | Accuracy | F1 Score |
| :---: | :---: | :---: | :---: | :---: |
| **1** | 0.235924 | 0.248634 | 0.904667 | 0.904901 |
| **2** | 0.201652 | 0.236843 | 0.908167 | 0.908053 |
| **3** | 0.130800 | 0.285101 | 0.908967 | 0.910079 |


#### 📊하드웨어 학습 효율 비교 (Step 5 최적화 효과)

| 지표 | 최초 학습 (Step 3) | Step 4, 5 통합 적용 후 | 개선 정도 |
| :--- | :---: | :---: | :---: |
| **전체 학습 시간 (Total Runtime)** | 3,382s (약 56분) | **2,527s (약 42분)** | **약 25.3% 단축** |
| **초당 처리 샘플 수 (Samples/sec)** | 106.4 | **142.4** | **약 33.8% 향상** |
| **총 학습 스텝 (Global Steps)** | 45,000 steps | **11,250 steps** | **배치 사이즈 확대 반영** |
| **평균 학습 손실 (Train Loss)** | 0.2404 | **0.2038** | **안정성 향상** |

[핵심 분석]
> Bucketing과 Dynamic Padding을 통해 불필요한 패딩(0) 연산을 획기적으로 줄이고 학습 속도와 자원 활용 능력 향상
> 초당 처리량(Throughput)이 33% 이상 증가
> 전체 학습 시간을 14분 절약


#### 📈 모델 성능 및 학습 안정성 비교 (Step 4 최적화 효과)

| 평가 지표 | 최초 학습 (Step 3) | Step 4, 5 통합 적용 후 | 개선 정도 |
| :--- | :---: | :---: | :---: |
| **최고 검증 정확도 (Best Val Acc)** | 0.9064 | **0.9089** | **+0.25%p 향상** |
| **최고 검증 F1 스코어 (Best Val F1)** | 0.9069 | **0.9100** | **+0.31%p 향상** |
| **최종 테스트 정확도 (Test Acc)** | 0.9065 | **0.9065** | **동일 수준 유지** |
| **학습 안정성 (Val Loss 추이)** | 0.27 → 0.42 (상승) | **0.24 → 0.28 (안정)** | **과적합 억제 성공** |

[학습 곡선 및 손실(Loss) 분석]
> 최초 학습: 1 Epoch 이후 Validation Loss가 0.27 → 0.34 → 0.42로 가파르게 상승하며 **과적합(Overfitting)**이 강하게 발생
> 최적화 학습: Validation Loss가 0.24 → 0.23 → 0.28 수준으로 훨씬 낮고 안정적으로 유지
> 해석: 배치 사이즈를 8에서 32로 키운 것이 그래디언트 안정화에 큰 기여를 했으며, 결과적으로 모델이 데이터의 노이즈에 덜 민감해지고 핵심 패턴을 더 잘 학습


#### [종합결론]

- 학습 시간을 25% 단축하면서도 성능 손실이 전혀 없음
- 질적인 성장: 정확도 수치 자체의 향상보다, Validation Loss의 안정화를 통해 모델의 신뢰도가 훨씬 높아짐.
- 한계점: Test 데이터셋 정확도가 90.65%에서 정체된 것으로 보아, 현재의 데이터셋(NSMC)과 모델 체급(koelectra-base)에서는 이 지점이 기술적인 임계치(Ceiling)에 가까워진 것으로 판단